In [1]:
from resources.imports import *

# Genetic Algorithm

In [ ]:
import numpy as np

def genetic_algorithm(
    population,
    nn_model,
    input_shape,
    fitness_fn,
    population_size=20,
    n_generations=100,
    mutation_rate=0.1,
    crossover_rate=0.5,
    disorder_bounds=(-0.2, 0.2) 
):
    # Initialize random population
    population = np.random.uniform(
        disorder_bounds[0], disorder_bounds[1], 
        size=(population_size,) + input_shape
    )
    
    for generation in range(n_generations):
        # Evaluate fitness (for each individual)
        fitness = np.array([fitness_fn(nn_model(indiv)) for indiv in population])
        
        # Select top individuals (elitism)
        idx = np.argsort(fitness)[::-1]
        population = population[idx]
        fitness = fitness[idx]
        
        # Keep top N as elites
        n_elite = population_size // 5
        new_population = population[:n_elite].copy()
        
        # Generate new individuals
        while len(new_population) < population_size:
            # Crossover: pick two parents, mix their genes
            if np.random.rand() < crossover_rate:
                parents = population[np.random.choice(population_size, 2, replace=False)]
                mask = np.random.rand(*input_shape) < 0.5
                child = np.where(mask, parents[0], parents[1])
            else:
                # No crossover, just copy parent
                child = population[np.random.randint(population_size)].copy()
            
            # Mutation
            mutation_mask = np.random.rand(*input_shape) < mutation_rate
            mutation = np.random.uniform(disorder_bounds[0], disorder_bounds[1], input_shape)
            child[mutation_mask] = mutation[mutation_mask]
            
            # Clip to bounds
            child = np.clip(child, disorder_bounds[0], disorder_bounds[1])
            new_population.append(child)
        
        population = np.array(new_population)
    
    # Return best individual
    best_idx = np.argmax([fitness_fn(nn_model(indiv)) for indiv in population])
    return population[best_idx]


In [ ]:
# Update text file with current iteration's results
def update_text_file(iteration, n_plies, V1D, V2D, V3D, weight, strains, fileName):
    with open(fileName, "a") as file:
        file.write("Iteration %d:\n" % iteration)
        file.write("Number of plies:\n")
        file.write(' '.join(str(i) for i in n_plies))
        file.write("\n")
        file.write("Lamination Parameters V1D:\n")
        file.write(' '.join(['%.4f' % i for i in V1D]))
        file.write("\n")
        file.write("Lamination Parameters V2D:\n")
        file.write(' '.join(['%.4f' % i for i in V2D]))
        file.write("\n")
        file.write("Lamination Parameters V3D:\n")
        file.write(' '.join(['%.4f' % i for i in V3D]))
        file.write("\n")
        file.write("Weight: %.3f\n" % weight)
        file.write("Abaqus outputs (E11, E22, E12, BU):\n")
        file.write(' '.join(['%.4f' % i for i in strains]))
        file.write("\n\n")

# Update text file with current iteration's results
def update_plot(iteration, weight_history):
    # print(iteration, weight_history, fitness_history)
    plt.clf()
    plt.plot(range(1, iteration + 1), weight_history, marker='o', linestyle='--')
    plt.xlabel('Generations')
    plt.ylabel('Weight (tonne)')
    # plt.title('Weight vs Generation')
    plt.grid(True)
    
    plt.tight_layout()
    plt.pause(0.1)

def run_abq_inp_with_processors(directory, num_processors):
    # get the input file name from the path
    inp_fnm = os.path.basename(directory)
    inp_fnm = os.path.join(inp_fnm, 'Job-1.inp')
    
    # Extract input file name without .inp --> job name
    job_name = "Job-1"
    abaqus_command = "abaqus job="+job_name+ " inp="+inp_fnm + " cpus=" + str(num_processors)
    try:
        subprocess.run(abaqus_command, shell=True, check=True)
        # print("Job {job_name} completed successfully with {num_processors} processors.")
    except subprocess.CalledProcessError as e:
        # print("Error running job {job_name} with {num_processors} processors.")
        print(e)

# Initialize FE solver
def fe_solver(individual):
    global iteration
    global weight_history
    
    iteration = iteration + 1
    
    if tuple(individual) in fitness_cache:
        # print("Unable to find new data")
        return fitness_cache[tuple(individual)]
    
    calv1d_history = []
    calv2d_history = []
    calv3d_history = []
    count = 0
    for ij in range(1,PANEL_NUM+1):
        n_pliesID = individual[count:count+3]
        ply_angle = generate_custom_array(n_pliesID)
        count = count + 3
        calv1d, calv2d, calv3d, check = generate_plies(ply_angle)
        calv1d_history.append(calv1d)
        calv2d_history.append(calv2d)
        calv3d_history.append(calv3d)
    
    if initialRuns.lower() == 'yes':
        # Placeholder FE solver evaluation, replace with real solver
        with open('numberOfPlies.txt', 'a') as f:
            np.savetxt(f, [individual], delimiter=' ', fmt='%d')
        
        if actualRun.lower() == 'yes':
            abaqus_command = "abaqus cae noGUI=ABAQUS_BUG_WING_MODEL_loft.py"
            done = subprocess.run(abaqus_command, shell=True, cwd=PATH, capture_output=True, text=True)
            # print("Output:\n", done.stdout)
            # print("Error:\n", done.stderr)
            
            postPro_command = "abaqus cae noGUI=postProcess.py"
            done = subprocess.run(postPro_command, shell=True, cwd=PATH, capture_output=True, text=True)
            # print("Output:\n", done.stdout)
            # print("Error:\n", done.stderr)
        else:
            abaqus_command = "abaqus cae noGUI=ABAQUS_SAMPLE.py"
            done = subprocess.run(abaqus_command, shell=True, cwd=PATH, capture_output=True, text=True)
            # print("Output:\n", done.stdout)
            # print("Error:\n", done.stderr)
            
            postPro_command = "abaqus cae noGUI=postProcess_SAMPLE.py"
            done = subprocess.run(postPro_command, shell=True, cwd=PATH, capture_output=True, text=True)
            # print("Output:\n", done.stdout)
            # print("Error:\n", done.stderr)
        
        data = read_line_as_array('abaqusResults.txt')
        weight = data[0]
        strains = [data[1], data[2], data[3], data[4]]
    else:
        data = read_line_as_array('abaqusResults.txt', line_number=iteration)
        weight = data[0]
        strains = [data[1], data[2], data[3], data[4]]
    
    # Check if strains violate constraints
    if data[1] > strain_limits[0] or data[2] > strain_limits[1] or data[3] > strain_limits[2] or data[4] < strain_limits[3]:
        weight = np.inf
        fitness_value = -np.inf # Penalize individuals violating constraints
        print("Failed Iteration: ", iteration, weight, fitness_value)
        # print("Strains Values: ", ALLOWABLE_STRAINS['E11'], E11, ALLOWABLE_STRAINS['E22'], E22, ALLOWABLE_STRAINS['E12'], E12)
        # print("Buckling Load: ", load)
    else:
        fitness_value = 1 / weight  # Higher fitness for lower weight
        print("Passed Iteration: ", iteration, weight, fitness_value)
        # print("Strains Values: ", ALLOWABLE_STRAINS['E11'], E11, ALLOWABLE_STRAINS['E22'], E22, ALLOWABLE_STRAINS['E12'], E12)
        # print("Buckling Load: ", load)
    
    fitness_cache[tuple(individual)] = fitness_value
    
    # Update result file
    update_text_file(iteration, individual, calv1d_history, calv2d_history, calv3d_history, weight, strains, fileName)
    
    # # Update history
    # weight_history.append(weight)
    
    # # Update plot
    # update_plot(iteration, weight_history)
    
    return fitness_value

# Tournament selection
def tournament_selection(population, fitness_values, tournament_ratio=0.5):
    tournament_size = int(len(population) * tournament_ratio)
    selected_population = []
    for _ in range(len(population)):
        # Select random indices for the tournament
        tournament_indices = np.random.choice(len(population), size=tournament_size, replace=False)
        # Get the fitness values of the selected tournament individuals
        tournament_fitness = [fitness_values[i] for i in tournament_indices]
        # Find the index of the winner in the original population
        winner_index = tournament_indices[np.argmax(tournament_fitness)]
        # Append the winner to the selected population
        selected_population.append(population[winner_index])
    return selected_population

# Crossover operation
def crossover(parent1, parent2):
    crossover_point = np.random.randint(2, NUM_GENES+2)  # Choose crossover point
    child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
    child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
    return child1, child2

def multipoint_crossover(parent1, parent2, num_points):
    """
    Perform multipoint crossover on two parents to produce two offspring.
    
    Parameters:
    - parent1 (list): The first parent individual (e.g., list of genes).
    - parent2 (list): The second parent individual (e.g., list of genes).
    - num_points (int): Number of crossover points. Defaults to 2.
    
    Returns:
    - child1, child2 (list): Two offspring produced by crossover.
    """
    # Ensure parents are of equal length
    if len(parent1) != len(parent2):
        raise ValueError("Parents must have the same length.")
    
    # Sort crossover points in increasing order
    crossover_points = sorted(random.sample(range(1, len(parent1)), num_points))
    
    # Initialize children as copies of the parents
    child1, child2 = parent1[:], parent2[:]
    
    # Alternate segments between parents for both children
    last_point = 0
    for i, point in enumerate(crossover_points):
        if i % 2 == 1:
            # Swap segments: child1 gets segment from parent2, and vice versa
            child1[last_point:point], child2[last_point:point] = parent2[last_point:point], parent1[last_point:point]
        # Move to the next segment
        last_point = point
    
    # Handle the last segment after the final crossover point
    if num_points % 2 == 1:
        child1[last_point:], child2[last_point:] = parent2[last_point:], parent1[last_point:]

    return child1, child2

# Mutation operation
def mutate(individual):
    mutation_indices = np.random.rand(NUM_GENES) < MUTATION_RATE
    mutation = np.zeros_like(individual)
    mutation[mutation_indices] = np.random.randint(-10, 0, size=np.count_nonzero(mutation_indices))
    individual += mutation
    individual = np.clip(individual, n_min, None)  # Clip values to be at least 1
    return individual

# Adaptive Mutation Function
def mutate_advanced(individual, termination_counter):
    if termination_counter == 0:
        current_mutation_rate = MUTATION_RATE  # Highest mutation rate
    elif 1 <= termination_counter < 3:
        current_mutation_rate = MUTATION_RATE / termination_counter  # Reduce mutation rate
    else:
        current_mutation_rate = MUTATION_RATE

    mutation_indices = np.random.rand(len(individual)) < current_mutation_rate
    mutation = np.zeros_like(individual)
    
    if termination_counter > 3:
        mutation[mutation_indices] = np.random.randint(1, 5, size=np.count_nonzero(mutation_indices))  # Adding values
    else:
        mutation[mutation_indices] = np.random.randint(-10, 0, size=np.count_nonzero(mutation_indices))  # Dropping values
    
    individual += mutation
    individual = np.clip(individual, n_min, None)  # Clip values within range
    return individual

def genetic_algorithm(population):
    
    best_fitness = -np.inf
    best_individual = None
    terminationCounter = 0
    for generation in range(NUM_GENERATIONS):

        # Evaluate fitness if not already calculated
        if tuple(map(tuple, population)) not in fitness_cache:
            fitness_values = np.array([fe_solver(individual) for individual in population])    

        generation_best_index = np.argmax(fitness_values)

        generation_best_fitness = fitness_values[generation_best_index]

        generation_best_individual = population[generation_best_index]

        # Update best solution found so far
        if generation_best_fitness > best_fitness:
            best_fitness = generation_best_fitness
            best_individual = copy.deepcopy(generation_best_individual)
            print("better solution found")
            terminationCounter = 0
        else:
            best_individual = copy.deepcopy(best_individual)
            print("better solution is not found")
            terminationCounter = terminationCounter + 1
        
        # Update history
        weight_history.append(1/best_fitness)
        
        # Update plot
        update_plot(generation+1, weight_history)
        
        calv1d_history = []
        calv2d_history = []
        calv3d_history = []
        count = 0
        for ij in range(1,PANEL_NUM+1):
            n_pliesID = best_individual[count:count+3]
            ply_angle = generate_custom_array(n_pliesID)
            count = count + 3
            calv1d, calv2d, calv3d, check = generate_plies(ply_angle)
            calv1d_history.append(calv1d)
            calv2d_history.append(calv2d)
            calv3d_history.append(calv3d)
        
        # Update text file from GA
        update_text_file(generation+1, best_individual, calv1d_history, calv2d_history, calv3d_history, 1/best_fitness, [0, 0, 0, 0], "GA_results.txt")
        
        
        # Selection (tournament selection)
        selected_population = tournament_selection(population, fitness_values)
        # Crossover
        for i in range(0, len(selected_population), 2):
            if np.random.rand() < CROSSOVER_RATE:
                if i + 1 < len(selected_population):  # Check if there is a pair
                    parent1, parent2 = selected_population[i], selected_population[i + 1]
                    population[i], population[i + 1] = multipoint_crossover(parent1, parent2, num_points=int(len(parent1)*0.2))
                
        # Mutation
        for i in range(POPULATION_SIZE):
            # population[i] = mutate_advanced(population[i], terminationCounter)
            population[i] = mutate(population[i])
                    
        # Check termination condition
        print("Generations:", generation, 1/best_fitness, 1/generation_best_fitness)
        print("Best individual", best_individual)
        
        if (generation_best_fitness < best_fitness) and (terminationCounter >= 0.2*NUM_GENERATIONS):
            return best_individual, best_fitness, generation
            
    return best_individual, best_fitness, generation

# Mutation Algorithm

In [4]:
def mutation_algorithm(
    nn_model,
    input_shape,
    fitness_fn,
    disorder_bounds=(-0.2, 0.2),
    n_iterations=1000,
    mutation_rate=0.05,
    step_size=0.02,
    n_restarts=10
):
    best_solution = None
    best_fitness = -np.inf
    
    for restart in range(n_restarts):
        # Start with random solution
        solution = np.random.uniform(disorder_bounds[0], disorder_bounds[1], input_shape)
        fitness = fitness_fn(nn_model(solution))
        
        for iter in range(n_iterations):
            # Mutate: random subset of elements
            mutation_mask = np.random.rand(*input_shape) < mutation_rate
            mutation = (np.random.uniform(-step_size, step_size, input_shape)) * mutation_mask
            candidate = np.clip(solution + mutation, disorder_bounds[0], disorder_bounds[1])
            candidate_fitness = fitness_fn(nn_model(candidate))
            
            # Accept if improved
            if candidate_fitness > fitness:
                solution = candidate
                fitness = candidate_fitness
        
        if fitness > best_fitness:
            best_fitness = fitness
            best_solution = solution
    
    return best_solution


# Simulated Annealing

In [5]:
def simulated_annealing(
    nn_model,
    input_shape,
    fitness_fn,
    disorder_bounds=(-0.2, 0.2),
    n_iterations=5000,
    initial_temp=1.0,
    final_temp=1e-3,
    step_size=0.02
):
    # Start with random solution
    solution = np.random.uniform(disorder_bounds[0], disorder_bounds[1], input_shape)
    fitness = fitness_fn(nn_model(solution))
    best_solution, best_fitness = solution, fitness
    
    temp = initial_temp
    temp_decay = (final_temp / initial_temp) ** (1.0 / n_iterations)
    
    for _ in range(n_iterations):
        # Propose a mutation
        mutation = np.random.uniform(-step_size, step_size, input_shape)
        candidate = np.clip(solution + mutation, disorder_bounds[0], disorder_bounds[1])
        candidate_fitness = fitness_fn(nn_model(candidate))
        
        # Decide whether to accept
        delta = candidate_fitness - fitness
        if delta > 0 or np.random.rand() < np.exp(delta / temp):
            solution = candidate
            fitness = candidate_fitness
            if fitness > best_fitness:
                best_fitness = fitness
                best_solution = solution
        
        temp *= temp_decay
    
    return best_solution
